<a href="https://colab.research.google.com/github/kalyan-1845/flyrank-ml-assignment/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kalyan-1845/flyrank-ml-assignment/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
I chose a Random Forest Classifier. It fits this lane because website traffic is non-linear and signals interact in complex ways (e.g., being in Position 2 vs Position 3 matters a lot more than Position 50 vs 51). Decision trees naturally handle these non-linear thresholds better than Logistic Regression.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
I am using GroupShuffleSplit on client_hash_id. If we used a standard random split, pages from the same website would end up in both the training and testing sets, which leaks website-specific patterns to the model. Group splitting ensures we train on a set of websites, and test on completely new websites the model has never seen.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report

# 1. Connect to Hugging Face
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

# 2. Get Features (Jan-March) and Target (April)
print("Downloading data and building features... (this takes ~30 seconds)")
query = f"""
    WITH windowed AS (
        SELECT content_hash_id,
               ANY_VALUE(client_hash_id) as client_id,
               SUM(CASE WHEN report_date > '2026-02-28' AND report_date <= '2026-03-31' THEN gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN report_date > '2026-01-31' AND report_date <= '2026-02-28' THEN gsc_impressions ELSE 0 END) AS imp_prev30,
               AVG(CASE WHEN report_date > '2026-02-28' AND report_date <= '2026-03-31' THEN gsc_avg_position END) AS pos_last30,
               SUM(CASE WHEN report_date >= '2026-04-01' THEN gsc_impressions ELSE 0 END) AS imp_future_target
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-0*/*.parquet')
        GROUP BY content_hash_id
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
"""
data = con.sql(query).df()
data['pos_last30'] = data['pos_last30'].fillna(100)

# 3. Define Label: Did it drop by more than 20% in the future?
data['is_declining'] = (data['imp_future_target'] < 0.8 * data['imp_last30']).astype(int)

# 4. Group Split (by Client)
feature_cols = ['imp_last30', 'imp_prev30', 'pos_last30']
X = data[feature_cols]
y = data['is_declining']
groups = data['client_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_tr, X_te, y_tr, y_te = X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx]

# 5. Train Model & Compare
model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
model.fit(X_tr, y_tr)
preds = model.predict(X_te)
baseline_preds = np.zeros(len(y_te)) # Baseline rule: assume nothing is declining

print("\n=== WEEK 4 BASELINE SCORE (Always guessing 'Not Declining') ===")
print(classification_report(y_te, baseline_preds, zero_division=0))

print("\n=== RANDOM FOREST MODEL SCORE ===")
print(classification_report(y_te, preds, zero_division=0))

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
Where is it wrong? The model struggles with "False Positives" on pages that had a massive viral spike recently. Because they spiked, they naturally decay back to normal traffic levels, which the model flags as a severe decline. What does it lean on? It heavily relies on the ratio between imp_last30 and imp_prev30 (momentum).

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.